In [7]:
import pandas as pd
import numpy as np
# Load dataset
df = pd.read_csv('/content/data.csv')
# Check loading
df.head()
# Check dataset dimensions
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])
print(df.columns.tolist())
# Target
y = df['exvalve']
print(y.head())
y = y.map({'Clean': 0, 'Dirty': 1})
print(y.value_counts())


Number of rows: 1000
Number of columns: 26
['id', 'rpm', 'motor_power', 'torque', 'outlet_pressure_bar', 'air_flow', 'noise_db', 'outlet_temp', 'wpump_outlet_press', 'water_inlet_temp', 'water_outlet_temp', 'wpump_power', 'water_flow', 'oilpump_power', 'oil_tank_temp', 'gaccx', 'gaccy', 'gaccz', 'haccx', 'haccy', 'haccz', 'bearings', 'wpump', 'radiator', 'exvalve', 'acmotor']
0    Clean
1    Clean
2    Clean
3    Clean
4    Clean
Name: exvalve, dtype: object
exvalve
0    800
1    200
Name: count, dtype: int64


In [12]:
# 20 Pedictor Features
features = [
    'rpm',
    'motor_power',
    'torque',
    'outlet_pressure_bar',
    'air_flow',
    'noise_db',
    'outlet_temp',
    'wpump_outlet_press',
    'water_inlet_temp',
    'water_outlet_temp',
    'wpump_power',
    'water_flow',
    'oilpump_power',
    'oil_tank_temp',
    'gaccx',
    'gaccy',
    'gaccz',
    'haccx',
    'haccy',
    'haccz'
]
# Feature Matrix
X = df[features]
X.head()
# Verify X & Y
print("X shape:", X.shape)
print("y shape:", y.shape)


X shape: (1000, 20)
y shape: (1000,)


In [13]:
# Missing values
print("Missing values in X:")
print(X.isnull().sum())

print("\nMissing values in y:")
print(y.isnull().sum())

Missing values in X:
rpm                    0
motor_power            0
torque                 0
outlet_pressure_bar    0
air_flow               0
noise_db               0
outlet_temp            0
wpump_outlet_press     0
water_inlet_temp       0
water_outlet_temp      0
wpump_power            0
water_flow             0
oilpump_power          0
oil_tank_temp          0
gaccx                  0
gaccy                  0
gaccz                  0
haccx                  0
haccy                  0
haccz                  0
dtype: int64

Missing values in y:
0


In [15]:
# Split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)
# Verify
print("Training data:")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("\nTesting data:")
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

Training data:
X_train: (800, 20)
y_train: (800,)

Testing data:
X_test: (200, 20)
y_test: (200,)


In [16]:
# Verify Class distribution after Split
print("Original target distribution:")
print(y.value_counts())

print("\nTraining target distribution:")
print(y_train.value_counts())

print("\nTesting target distribution:")
print(y_test.value_counts())

Original target distribution:
exvalve
0    800
1    200
Name: count, dtype: int64

Training target distribution:
exvalve
0    640
1    160
Name: count, dtype: int64

Testing target distribution:
exvalve
0    160
1     40
Name: count, dtype: int64


In [19]:
# Feature Scaling
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
# Verify Scaling
print("Scaled training data shape:", X_train_scaled.shape)
print("Scaled testing data shape:", X_test_scaled.shape)
print(X_train_scaled[:5])

Scaled training data shape: (800, 20)
Scaled testing data shape: (200, 20)
[[-0.7124396  -0.78542497 -0.52028962 -0.69495348 -0.52466762 -0.98945934
  -0.30047341 -0.0205859  -0.31766523 -0.1603637   0.7186207  -2.16238309
  -0.80579217 -0.43236325  0.31507155  1.31701355 -0.7170581   0.1247763
   0.03074751 -0.71748509]
 [ 1.39644789 -0.14549549 -1.32921598 -1.01578903  1.30505978  0.98433603
   1.24422937  2.11308498  1.48036777  1.65552432  0.22695392 -1.6327695
  -1.16613455  1.25230471 -0.87537571  0.18059712 -1.12266455 -0.94603476
  -0.81543763 -1.17671829]
 [ 0.69253928  1.6793519   1.08119586  1.38724769  0.97092283  0.39827412
   1.59642197  1.01666543  1.47126872  1.80446604  1.04652261 -0.2729189
   0.80373268  1.63885771 -0.60636903 -0.51434639  1.36509574 -0.71901037
  -0.08083853  1.30957216]
 [-0.00569265  0.62313068  0.86296967  0.95390437 -1.06967845 -0.43189595
   0.20808385  0.16441238 -0.08160697 -0.02055648  0.27651575  0.72134642
  -0.0578448  -0.16348374 -0.4463

In [23]:
# Logistic Regression
from sklearn.linear_model import LogisticRegression
# Create the model
logistic_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)
# Train
logistic_model.fit(X_train_scaled, y_train)
y_pred_lr = logistic_model.predict(X_test_scaled)
y_prob_lr = logistic_model.predict_proba(X_test_scaled)[:, 1]
# Import  Metrices
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef
)
# Calculate Metrices
accuracy_lr = accuracy_score(y_test, y_pred_lr)

auc_lr = roc_auc_score(y_test, y_prob_lr)

precision_lr = precision_score(y_test, y_pred_lr)

recall_lr = recall_score(y_test, y_pred_lr)

f1_lr = f1_score(y_test, y_pred_lr)

mcc_lr = matthews_corrcoef(y_test, y_pred_lr)
# Display
print("Logistic Regression Results")
print("---------------------------")
print("Accuracy :", accuracy_lr)
print("AUC      :", auc_lr)
print("Precision:", precision_lr)
print("Recall   :", recall_lr)
print("F1 Score :", f1_lr)
print("MCC      :", mcc_lr)
# Confusion Matrix
from sklearn.metrics import confusion_matrix

cm_lr = confusion_matrix(y_test, y_pred_lr)

print("Confusion Matrix:")
print(cm_lr)
# Classification Report
from sklearn.metrics import classification_report

print(classification_report(
    y_test,
    y_pred_lr,
    target_names=["Clean", "Dirty"]
))

Logistic Regression Results
---------------------------
Accuracy : 1.0
AUC      : 1.0
Precision: 1.0
Recall   : 1.0
F1 Score : 1.0
MCC      : 1.0
Confusion Matrix:
[[160   0]
 [  0  40]]
              precision    recall  f1-score   support

       Clean       1.00      1.00      1.00       160
       Dirty       1.00      1.00      1.00        40

    accuracy                           1.00       200
   macro avg       1.00      1.00      1.00       200
weighted avg       1.00      1.00      1.00       200



In [24]:
# Descision Tree
from sklearn.tree import DecisionTreeClassifier

dt_model = DecisionTreeClassifier(
    random_state=42
)

dt_model.fit(X_train, y_train)
#
y_pred_dt = dt_model.predict(X_test)
y_prob_dt = dt_model.predict_proba(X_test)[:, 1]
# Cal metrices
accuracy_dt = accuracy_score(y_test, y_pred_dt)
auc_dt = roc_auc_score(y_test, y_prob_dt)
precision_dt = precision_score(y_test, y_pred_dt)
recall_dt = recall_score(y_test, y_pred_dt)
f1_dt = f1_score(y_test, y_pred_dt)
mcc_dt = matthews_corrcoef(y_test, y_pred_dt)

print("Decision Tree Results")
print("---------------------")
print("Accuracy :", accuracy_dt)
print("AUC      :", auc_dt)
print("Precision:", precision_dt)
print("Recall   :", recall_dt)
print("F1 Score :", f1_dt)
print("MCC      :", mcc_dt)

# Confusion Matrix
cm_dt = confusion_matrix(y_test, y_pred_dt)

print("Confusion Matrix:")
print(cm_dt)

Decision Tree Results
---------------------
Accuracy : 0.995
AUC      : 0.9968750000000001
Precision: 0.975609756097561
Recall   : 1.0
F1 Score : 0.9876543209876543
MCC      : 0.9846381036309488
Confusion Matrix:
[[159   1]
 [  0  40]]


In [25]:
# KNN
from sklearn.neighbors import KNeighborsClassifier
knn_model = KNeighborsClassifier(
    n_neighbors=5
)

knn_model.fit(X_train_scaled, y_train)
# Predictions
y_pred_knn = knn_model.predict(X_test_scaled)

y_prob_knn = knn_model.predict_proba(X_test_scaled)[:, 1]

# Metrices
accuracy_knn = accuracy_score(y_test, y_pred_knn)
auc_knn = roc_auc_score(y_test, y_prob_knn)
precision_knn = precision_score(y_test, y_pred_knn)
recall_knn = recall_score(y_test, y_pred_knn)
f1_knn = f1_score(y_test, y_pred_knn)
mcc_knn = matthews_corrcoef(y_test, y_pred_knn)

print("KNN Results")
print("-----------")
print("Accuracy :", accuracy_knn)
print("AUC      :", auc_knn)
print("Precision:", precision_knn)
print("Recall   :", recall_knn)
print("F1 Score :", f1_knn)
print("MCC      :", mcc_knn)

# Confusion Matrix
cm_knn = confusion_matrix(y_test, y_pred_knn)

print("Confusion Matrix:")
print(cm_knn)

KNN Results
-----------
Accuracy : 0.945
AUC      : 0.9645312500000001
Precision: 1.0
Recall   : 0.725
F1 Score : 0.8405797101449275
MCC      : 0.8236276908284563
Confusion Matrix:
[[160   0]
 [ 11  29]]


In [26]:
#
from sklearn.naive_bayes import GaussianNB
# Creating & Training
nb_model = GaussianNB()

nb_model.fit(X_train_scaled, y_train)
# Prediction
y_pred_nb = nb_model.predict(X_test_scaled)

y_prob_nb = nb_model.predict_proba(X_test_scaled)[:, 1]

# Metrices
accuracy_nb = accuracy_score(y_test, y_pred_nb)
auc_nb = roc_auc_score(y_test, y_prob_nb)
precision_nb = precision_score(y_test, y_pred_nb)
recall_nb = recall_score(y_test, y_pred_nb)
f1_nb = f1_score(y_test, y_pred_nb)
mcc_nb = matthews_corrcoef(y_test, y_pred_nb)

print("Gaussian Naive Bayes Results")
print("----------------------------")
print("Accuracy :", accuracy_nb)
print("AUC      :", auc_nb)
print("Precision:", precision_nb)
print("Recall   :", recall_nb)
print("F1 Score :", f1_nb)
print("MCC      :", mcc_nb)

# Confusion Matrix
cm_nb = confusion_matrix(y_test, y_pred_nb)

print("Confusion Matrix:")
print(cm_nb)

Gaussian Naive Bayes Results
----------------------------
Accuracy : 0.825
AUC      : 0.92234375
Precision: 0.5362318840579711
Recall   : 0.925
F1 Score : 0.6788990825688074
MCC      : 0.6100533275926904
Confusion Matrix:
[[128  32]
 [  3  37]]


In [27]:
# Radom Forest
from sklearn.ensemble import RandomForestClassifier
# Create and train
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf_model.fit(X_train, y_train)

#Prediction
y_pred_rf = rf_model.predict(X_test)

y_prob_rf = rf_model.predict_proba(X_test)[:, 1]

# Metrices
accuracy_rf = accuracy_score(y_test, y_pred_rf)
auc_rf = roc_auc_score(y_test, y_prob_rf)
precision_rf = precision_score(y_test, y_pred_rf)
recall_rf = recall_score(y_test, y_pred_rf)
f1_rf = f1_score(y_test, y_pred_rf)
mcc_rf = matthews_corrcoef(y_test, y_pred_rf)

print("Random Forest Results")
print("---------------------")
print("Accuracy :", accuracy_rf)
print("AUC      :", auc_rf)
print("Precision:", precision_rf)
print("Recall   :", recall_rf)
print("F1 Score :", f1_rf)
print("MCC      :", mcc_rf)

# Confusion Matrix
cm_rf = confusion_matrix(y_test, y_pred_rf)

print("Confusion Matrix:")
print(cm_rf)


Random Forest Results
---------------------
Accuracy : 0.995
AUC      : 1.0
Precision: 1.0
Recall   : 0.975
F1 Score : 0.9873417721518988
MCC      : 0.9843495818960264
Confusion Matrix:
[[160   0]
 [  1  39]]


In [28]:
# Diagnostic — Logistic Regression without air_flow
# remove air flow
features_no_airflow = [
    'rpm',
    'motor_power',
    'torque',
    'outlet_pressure_bar',
    'noise_db',
    'outlet_temp',
    'wpump_outlet_press',
    'water_inlet_temp',
    'water_outlet_temp',
    'wpump_power',
    'water_flow',
    'oilpump_power',
    'oil_tank_temp',
    'gaccx',
    'gaccy',
    'gaccz',
    'haccx',
    'haccy',
    'haccz'
]
# Reduce Feature matrix
X_no_airflow = df[features_no_airflow]
# Train test split
X_train_no_airflow, X_test_no_airflow, y_train_no_airflow, y_test_no_airflow = train_test_split(
    X_no_airflow,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)
# Scale reduced featute
scaler_no_airflow = StandardScaler()

X_train_no_airflow_scaled = scaler_no_airflow.fit_transform(
    X_train_no_airflow
)

X_test_no_airflow_scaled = scaler_no_airflow.transform(
    X_test_no_airflow
)
# Train
logistic_no_airflow = LogisticRegression(
    max_iter=1000,
    random_state=42
)

logistic_no_airflow.fit(
    X_train_no_airflow_scaled,
    y_train_no_airflow
)
# Predict
y_pred_no_airflow = logistic_no_airflow.predict(
    X_test_no_airflow_scaled
)

y_prob_no_airflow = logistic_no_airflow.predict_proba(
    X_test_no_airflow_scaled
)[:, 1]
# 6 Metrices
accuracy_no_airflow = accuracy_score(
    y_test_no_airflow,
    y_pred_no_airflow
)

auc_no_airflow = roc_auc_score(
    y_test_no_airflow,
    y_prob_no_airflow
)

precision_no_airflow = precision_score(
    y_test_no_airflow,
    y_pred_no_airflow
)

recall_no_airflow = recall_score(
    y_test_no_airflow,
    y_pred_no_airflow
)

f1_no_airflow = f1_score(
    y_test_no_airflow,
    y_pred_no_airflow
)

mcc_no_airflow = matthews_corrcoef(
    y_test_no_airflow,
    y_pred_no_airflow
)

print("Logistic Regression WITHOUT air_flow")
print("------------------------------------")
print("Accuracy :", accuracy_no_airflow)
print("AUC      :", auc_no_airflow)
print("Precision:", precision_no_airflow)
print("Recall   :", recall_no_airflow)
print("F1 Score :", f1_no_airflow)
print("MCC      :", mcc_no_airflow)

# Confusion Matrix
cm_no_airflow = confusion_matrix(
    y_test_no_airflow,
    y_pred_no_airflow
)

print("Confusion Matrix:")
print(cm_no_airflow)

Logistic Regression WITHOUT air_flow
------------------------------------
Accuracy : 0.79
AUC      : 0.8587499999999999
Precision: 0.46153846153846156
Recall   : 0.3
F1 Score : 0.36363636363636365
MCC      : 0.25274793921627237
Confusion Matrix:
[[146  14]
 [ 28  12]]


In [29]:
test_data = X_test.copy()

test_data['exvalve'] = y_test.values

test_data.to_csv(
    'test_data.csv',
    index=False
)

print("test_data.csv created successfully")
print("Rows:", test_data.shape[0])
print("Columns:", test_data.shape[1])

test_data.csv created successfully
Rows: 200
Columns: 21
